In [1]:
import chromadb

In [2]:
# 创建客户端
client = chromadb.Client()

In [3]:
# 创建集合
collection1 = client.get_or_create_collection(name="collection1")
collection2 = client.get_or_create_collection(name="collection2")

## 新增数据（C）

In [5]:
# 1. 添加文本数据
collection1.add(
    ids=["1", "2", "3"],
    documents=[
        "This is the first document",
        "This is the second document",
        "This is the third document",
    ],
    metadatas=[
        {'chapter': 3, 'verse': 16},
        {'chapter': 3, 'verse': 5},
        {'chapter': 15, 'verse': 12}
    ]
)

In [26]:
collection1.add(
    ids=["4", "5", "6"],
    documents=[
        "Hello world!",
        "Goodbye world!",
        "Hello Chroma!",
    ],
    metadatas=[
        {'chapter': 4},
        {'chapter': 8},
        {'chapter': 11, 'verse': 10}
    ]
)

In [27]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6'],
 'embeddings': array([[-0.05846756,  0.07448348,  0.02305872, ...,  0.0695198 ,
          0.11604054,  0.05364618],
        [-0.061316  ,  0.08817796,  0.02569242, ...,  0.06425829,
          0.09380953,  0.04365631],
        [-0.05396889,  0.04353064, -0.02860478, ...,  0.02393956,
          0.09608021,  0.05749849],
        [-0.02038693,  0.02528078, -0.00056626, ..., -0.03065235,
          0.03755685,  0.03742788],
        [ 0.03618988,  0.07483862,  0.02247376, ..., -0.02649771,
         -0.02245256, -0.02141615],
        [-0.10963725,  0.0304918 ,  0.01641893, ...,  0.00313875,
          0.05246634, -0.02596475]]),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world!',
  'Goodbye world!',
  'Hello Chroma!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter': 3, 'verse': 16},
  {'verse': 5, 'chapter': 3},
  {'c

In [7]:
collection1.peek()['embeddings'].shape

(3, 384)

In [11]:
# 2. 添加数据的同时，指定嵌入向量
collection2.add(
    ids=["1", "2", "3"],
    documents=[
        "This is the first document",
        "This is the second document",
        "This is the third document",
    ],
    embeddings=[
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0]
    ]
)

In [13]:
collection2.peek()

{'ids': ['1', '2', '3'],
 'embeddings': array([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]]),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None, None, None]}

In [14]:
# collection2.add(
#     ids="4",
#     documents= "This is the fourth document",
# )

InvalidArgumentError: Collection expecting embedding with dimension of 3, got 384

## 查询数据（R）

In [16]:
# 1. 直接获取数据
collection1.get(ids=["2", "3"], include=["metadatas", "documents", "embeddings"])

{'ids': ['2', '3'],
 'embeddings': array([[-6.13159984e-02,  8.81779566e-02,  2.56924219e-02,
          2.88427789e-02,  2.98360549e-02,  1.40053844e-02,
         -3.17784138e-02,  1.60764735e-02,  2.54136827e-02,
          7.86870020e-04,  4.16737832e-02,  7.78555423e-02,
         -4.99316026e-03, -3.92468385e-02, -5.77454343e-02,
          1.92843229e-02, -4.96807396e-02, -6.30024308e-03,
          3.50779481e-02,  5.36748432e-02,  5.07013090e-02,
          6.13464452e-02,  3.10037583e-02, -3.32532898e-02,
          1.34150870e-02,  6.73581883e-02, -1.07689925e-01,
          5.52609339e-02,  4.11785841e-02, -8.33426192e-02,
          2.40026899e-02,  6.34173453e-02, -8.38173181e-03,
          8.88877641e-03,  3.83739844e-02, -5.77810667e-02,
          8.14787075e-02,  2.19539404e-02,  2.92267054e-02,
          2.35394407e-02,  1.69644989e-02, -1.05684839e-01,
         -4.40488830e-02, -1.46725010e-02, -1.13239035e-03,
          3.50902118e-02, -5.47159202e-02,  7.12127890e-03,
      

In [22]:
# 2. 增加筛选条件
collection1.get(
    where={'chapter': {"$gte": 3}},
    where_document={'$contains': 'then'}
)

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [30]:
# 3. 相似查询
# 3.1 直接传入数据（文本）
collection1.query(
    query_texts="hello",
    where={"chapter": {"$gte": 4}},
    n_results=3
)

{'ids': [['4', '6', '5']],
 'embeddings': None,
 'documents': [['Hello world!', 'Hello Chroma!', 'Goodbye world!']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'chapter': 4}, {'chapter': 11, 'verse': 10}, {'chapter': 8}]],
 'distances': [[0.5710345506668091, 0.8545365333557129, 1.330894947052002]]}

In [32]:
# 3.2 传入向量
collection2.query(
    query_embeddings=[1.1, 2.3, 3.4],
    include=["metadatas", "documents", "embeddings", "distances"]
)

{'ids': [['1', '2', '3']],
 'embeddings': [array([[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]])],
 'documents': [['This is the first document',
   'This is the second document',
   'This is the third document']],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.2600000500679016, 22.459999084472656, 98.66000366210938]]}

## 更新数据（U）

In [33]:
# 插入相同id的数据，不会改变
collection1.add(
    ids=["4", "5", "6"],
    documents=[
        "Hello world 111!",
        "Goodbye world 111!",
        "Hello Chroma 111!",
    ],
    metadatas=[
        {'chapter': 4},
        {'chapter': 5},
        {'chapter': 12, 'verse': 13}
    ]
)

In [34]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6'],
 'embeddings': array([[-0.05846756,  0.07448348,  0.02305872, ...,  0.0695198 ,
          0.11604054,  0.05364618],
        [-0.061316  ,  0.08817796,  0.02569242, ...,  0.06425829,
          0.09380953,  0.04365631],
        [-0.05396889,  0.04353064, -0.02860478, ...,  0.02393956,
          0.09608021,  0.05749849],
        [-0.02038693,  0.02528078, -0.00056626, ..., -0.03065235,
          0.03755685,  0.03742788],
        [ 0.03618988,  0.07483862,  0.02247376, ..., -0.02649771,
         -0.02245256, -0.02141615],
        [-0.10963725,  0.0304918 ,  0.01641893, ...,  0.00313875,
          0.05246634, -0.02596475]]),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world!',
  'Goodbye world!',
  'Hello Chroma!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter': 3, 'verse': 16},
  {'verse': 5, 'chapter': 3},
  {'v

In [35]:
# update更改指定id的数据
collection1.update(
    ids=["4", "5", "6"],
    documents=[
        "Hello world 111!",
        "Goodbye world 111!",
        "Hello Chroma 111!",
    ],
    metadatas=[
        {'chapter': 4},
        {'chapter': 5},
        {'chapter': 12, 'verse': 13}
    ]
)

In [36]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6'],
 'embeddings': array([[-0.05846756,  0.07448348,  0.02305872, ...,  0.0695198 ,
          0.11604054,  0.05364618],
        [-0.061316  ,  0.08817796,  0.02569242, ...,  0.06425829,
          0.09380953,  0.04365631],
        [-0.05396889,  0.04353064, -0.02860478, ...,  0.02393956,
          0.09608021,  0.05749849],
        [-0.08065048,  0.03005748, -0.00960962, ...,  0.00635418,
         -0.06554214, -0.05310861],
        [-0.01924923,  0.08375506,  0.00162334, ..., -0.01317479,
         -0.08101652, -0.07411238],
        [-0.15383206,  0.03568975, -0.01151466, ...,  0.02735918,
         -0.01315342, -0.06161033]]),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world 111!',
  'Goodbye world 111!',
  'Hello Chroma 111!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'verse': 16, 'chapter': 3},
  {'verse': 5, 'chapter

In [43]:
# upsert = update + insert
# update更改指定id的数据
collection1.upsert(
    ids=["4", "5", "6", "7"],
    documents=[
        "Hello world 222!",
        "Goodbye world 222!",
        "Hello Chroma 222!",
        "Hello Chroma 22222!",
    ],
    metadatas=[
        {'chapter': 1},
        {'chapter': 1},
        {'chapter': 1},
        {'chapter': 1, 'price': 50.0},
    ]
)

In [44]:
collection1.peek()

{'ids': ['1', '2', '3', '4', '5', '6', '7'],
 'embeddings': array([[-0.05846756,  0.07448348,  0.02305872, ...,  0.0695198 ,
          0.11604054,  0.05364618],
        [-0.061316  ,  0.08817796,  0.02569242, ...,  0.06425829,
          0.09380953,  0.04365631],
        [-0.05396889,  0.04353064, -0.02860478, ...,  0.02393956,
          0.09608021,  0.05749849],
        ...,
        [ 0.00859754,  0.03723701, -0.00891266, ..., -0.06948347,
         -0.10250814, -0.06325991],
        [-0.13393998, -0.00590101, -0.01544944, ...,  0.01599906,
         -0.0122663 , -0.05547561],
        [-0.13512173, -0.00360029, -0.00462356, ...,  0.01359506,
          0.00168625, -0.05872585]]),
 'documents': ['This is the first document',
  'This is the second document',
  'This is the third document',
  'Hello world 222!',
  'Goodbye world 222!',
  'Hello Chroma 222!',
  'Hello Chroma 22222!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter

## 删除数据（D）

In [45]:
collection1.delete(ids="2")

In [46]:
collection1.peek()

{'ids': ['1', '3', '4', '5', '6', '7'],
 'embeddings': array([[-0.05846756,  0.07448348,  0.02305872, ...,  0.0695198 ,
          0.11604054,  0.05364618],
        [-0.05396889,  0.04353064, -0.02860478, ...,  0.02393956,
          0.09608021,  0.05749849],
        [-0.04155245, -0.0236222 , -0.01487361, ..., -0.03260467,
         -0.0798054 , -0.04597215],
        [ 0.00859754,  0.03723701, -0.00891266, ..., -0.06948347,
         -0.10250814, -0.06325991],
        [-0.13393998, -0.00590101, -0.01544944, ...,  0.01599906,
         -0.0122663 , -0.05547561],
        [-0.13512173, -0.00360029, -0.00462356, ...,  0.01359506,
          0.00168625, -0.05872585]]),
 'documents': ['This is the first document',
  'This is the third document',
  'Hello world 222!',
  'Goodbye world 222!',
  'Hello Chroma 222!',
  'Hello Chroma 22222!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'verse': 16, 'chapter': 3},
  {'verse': 12, 'chapter': 15},

In [47]:
collection1.delete(ids=["2", "3", "4"])

In [48]:
collection1.peek()

{'ids': ['1', '5', '6', '7'],
 'embeddings': array([[-0.05846756,  0.07448348,  0.02305872, ...,  0.0695198 ,
          0.11604054,  0.05364618],
        [ 0.00859754,  0.03723701, -0.00891266, ..., -0.06948347,
         -0.10250814, -0.06325991],
        [-0.13393998, -0.00590101, -0.01544944, ...,  0.01599906,
         -0.0122663 , -0.05547561],
        [-0.13512173, -0.00360029, -0.00462356, ...,  0.01359506,
          0.00168625, -0.05872585]]),
 'documents': ['This is the first document',
  'Goodbye world 222!',
  'Hello Chroma 222!',
  'Hello Chroma 22222!'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter': 3, 'verse': 16},
  {'chapter': 1},
  {'verse': 13, 'chapter': 1},
  {'chapter': 1, 'price': 50.0}]}